In this notebook, I will clean and prepare the University of Cambridge construction dataset for my machine learning model. Since the JSON file is quite large (1.29 GB), I will process it in smaller parts to avoid using too much computer memory. I will extract individual construction activities, check for incorrect or missing duration values, keep track of removed records, and save the cleaned data for further checks, feature engineering, and splitting the data by project.

### Install dependencies

In [ ]:
!pip -q install orjson pandas numpy matplotlib py7zr

In [ ]:
import shutil

In [ ]:
from pathlib import Path
from google.colab import files

In [ ]:
import zipfile, re, json, math, csv
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import orjson

### Verify Extracted Dataset and Setup Output Directories

In [ ]:
import os
from pathlib import Path

# Automatically check for the existing .7z file in the workspace
archive_path = Path('/content/JPF_Anonymised_Project_Data.7z')

if archive_path.exists():
    print(f"Found archive file: {archive_path}")
    # Extracting the 7z file to /content/ extracted folder
    print("Extracting archive... Please wait.")
    !7z x {archive_path} -o/content/extracted -y > /dev/null
    print("Extraction completed. Contents:")
    !ls -la /content/extracted
else:
    raise FileNotFoundError(f"{archive_path} not found. Please upload the 7z dataset file to /content/")

Found archive file: /content/JPF_Anonymised_Project_Data.7z
Extracting archive... Please wait.
ERROR: /content/JPF_Anonymised_Project_Data.7z
/content/JPF_Anonymised_Project_Data.7z
Open ERROR: Cannot open the file as [7z] archive


ERRORS:
Unexpected end of archive
Extraction completed. Contents:
ls: cannot access '/content/extracted': No such file or directory


In [9]:
from google.colab import files

print("Select your 'JPF_Anonymised_Project_Data.7z' file to upload:")
uploaded = files.upload()

for name in uploaded.keys():
    print(f"Successfully uploaded: {name}")

Select your 'JPF_Anonymised_Project_Data.7z' file to upload:


KeyboardInterrupt: 

In [ ]:
json_path = '/content/extracted/JPF_Anonymised_Project_Data.json'

# Read the beginning of the file to inspect the structure
with open(json_path, 'r', encoding='utf-8') as f:
    preview = f.read(1000)

print("=== JSON File Preview (First 1000 characters) ===")
print(preview)

FileNotFoundError: [Errno 2] No such file or directory: '/content/extracted/JPF_Anonymised_Project_Data.json'

In [ ]:
import orjson
import pandas as pd
import numpy as np

# Load and parse using orjson for high speed and memory efficiency
print("Loading JSON file...")
with open(json_path, 'rb') as f:
    data = orjson.loads(f.read())

print("Parsing and flattening activities...")
records = []
projects_data = data.get("Projects", {})

for proj_id, proj_info in projects_data.items():
    proj_meta = {
        "Project_ID": proj_info.get("Project_ID"),
        "UK_Region": proj_info.get("UK_Region"),
        "Project_Type": proj_info.get("Project_Type"),
        "Earliest_Project_Start_Date": proj_info.get("Earliest_Project_Start_Date"),
        "Status_Data_Date": proj_info.get("Status_Data_Date"),
        "Latest_Project_Finish_Date": proj_info.get("Latest_Project_Finish_Date")
    }

    activities = proj_info.get("Activities", {})
    for act_id, act_info in activities.items():
        record = proj_meta.copy()
        record.update({
            "Task_ID": act_info.get("Task_ID"),
            "Activity_Description": act_info.get("Activity_Description"),
            "WBS_ID": act_info.get("WBS_ID"),
            "WBS_Description": act_info.get("WBS_Description"),
            "Original_Duration_Estimate_ODE": act_info.get("Original_Duration_Estimate(ODE)"),
            "At_Completion_Duration": act_info.get("At_Completion_Duration"),
            "Duration_Variance": act_info.get("Duration_Variance"),
            "Actual_Duration": act_info.get("Actual_Duration")
        })
        records.append(record)

df = pd.DataFrame(records)
print(f"Initial Load Completed. Total records extracted: {len(df)}")

# Convert duration columns to numeric
duration_cols = ["Original_Duration_Estimate_ODE", "At_Completion_Duration", "Duration_Variance", "Actual_Duration"]
for col in duration_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Validate and Clean Durations (Remove rows where key duration targets are negative or NaN)
initial_count = len(df)
# Dropping rows with missing ODE or non-positive durations
valid_mask = (df["Original_Duration_Estimate_ODE"].notna()) & (df["Original_Duration_Estimate_ODE"] > 0)
cleaned_df = df[valid_mask].copy()
removed_count = initial_count - len(cleaned_df)

print(f"Cleaned Dataset. Removed {removed_count} records with non-positive or missing Original Durations.")
print(f"Remaining cleaned records: {len(cleaned_df)}")
cleaned_df.head()

In [ ]:
OUTPUT_DIR = Path('/content/construction_ml_preprocessing')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Reference the already extracted JSON file
json_file = Path(json_path)
if json_file.exists():
    print('JSON file path:', json_file)
    print(f'Uncompressed size: {json_file.stat().st_size / (1024**3):.2f} GiB')
else:
    print(f"Expected JSON file not found at: {json_file}")

print('Output directory:', OUTPUT_DIR)

### Parse project objects incrementally and export activity fields

In [ ]:
RAW_COLUMNS = [
    'project_id', 'activity_id', 'project_type', 'uk_region',
    'original_duration', 'at_completion_duration', 'reported_growth',
    'physical_percent_complete', 'planned_start', 'planned_finish',
    'actual_start', 'actual_finish', 'gmm_topic', 'task_type',
    'concurrent_activity_count', 'project_status_date'
]

# This exact-indentation parser follows the formatting in the Cambridge JPF JSON.
# It avoids interpreting nested Activity_Relationships as individual projects.
PROJECT_OPEN = re.compile(rb'^ {12}"([^"]+)": \{\s*$')
PROJECT_CLOSE = re.compile(rb'^ {12}\},?\s*$')

def numeric(value):
    if value is None or isinstance(value, bool):
        return None
    try:
        value = float(value)
        return value if math.isfinite(value) else None
    except (TypeError, ValueError):
        return None

def extract_project(key, data, writer, stats):
    obj = orjson.loads(data)
    project_id = str(obj.get('Project_ID') or key)
    activities = obj.get('Activities') or {}
    if not isinstance(activities, dict):
        stats['projects_invalid_activities_container'] += 1
        return
    stats['projects'] += 1
    for activity_key, a in activities.items():
        stats['activities'] += 1
        writer.writerow({
            'project_id': project_id,
            'activity_id': str(a.get('Task_ID') or activity_key),
            'project_type': obj.get('Project_Type'),
            'uk_region': obj.get('UK_Region'),
            'original_duration': numeric(a.get('Original_Duration_Estimate(ODE)')),
            'at_completion_duration': numeric(a.get('At_Completion_Duration')),
            'reported_growth': numeric(a.get('Duration_Growth_Percentage(Y_values)')),
            'physical_percent_complete': numeric(a.get('Physical_Percent_Complete')),
            'planned_start': a.get('Planned_Start_Date'),
            'planned_finish': a.get('Planned_Finish_Date'),
            'actual_start': a.get('Actual_Start_Date'),
            'actual_finish': a.get('Actual_Finish_Date'),
            'gmm_topic': a.get('GMM_Topic_Classification'),
            'task_type': a.get('Task_Type:'),
            'concurrent_activity_count': numeric(a.get('Concurrent_Activity_Count')),
            'project_status_date': obj.get('Status_Data_Date')
        })

raw_csv = OUTPUT_DIR / 'activity_audit.csv'
stats = Counter()

# Open the extracted JSON file directly instead of trying to read from a zip_path
with open(json_path, 'rb') as stream, \
     raw_csv.open('w', encoding='utf-8', newline='') as out:
    writer = csv.DictWriter(out, fieldnames=RAW_COLUMNS)
    writer.writeheader()
    current_key, parts = None, []
    in_projects = False
    for line in stream:
        if not in_projects:
            if line.strip() == b'"Projects": {':
                in_projects = True
            continue
        if current_key is None:
            match = PROJECT_OPEN.match(line)
            if match:
                current_key = match.group(1).decode('utf-8')
                parts = [b'{']
            elif line.startswith(b'      }') and line.strip() in (b'}', b'},'):
                break
        else:
            if PROJECT_CLOSE.match(line):
                parts.append(b'}')
                extract_project(current_key, b'\n'.join(parts), writer, stats)
                current_key, parts = None, []
            else:
                parts.append(line)

assert stats['projects'] > 0 and stats['activities'] > 0, 'Failed to parse Projects section'
print('Projects parsed:', f"{stats['projects']:,}")
print('Activity records:', f"{stats['activities']:,}")
print('Saved:', raw_csv)

### Load the extracted activity table and check its shape

In [ ]:
df = pd.read_csv(
    raw_csv,
    dtype={'project_id': 'string', 'activity_id': 'string',
           'project_type': 'string', 'uk_region': 'string',
           'gmm_topic': 'string', 'task_type': 'string'},
    low_memory=False
)
print('Shape:', df.shape)
print('Number of unique projects:', df['project_id'].nunique())
print('Duplicated (project_id, activity_id) combinations:',
      df.duplicated(['project_id', 'activity_id']).sum())
display(df.head())

In [ ]:
num_cols = ['original_duration', 'at_completion_duration',
            'reported_growth', 'physical_percent_complete',
            'concurrent_activity_count']
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

for c in ['planned_start', 'planned_finish', 'actual_start', 'actual_finish', 'project_status_date']:
    df[c + '_parsed'] = pd.to_datetime(df[c], errors='coerce')

# Each flag is reported separately; records can fail multiple checks.
df['valid_original'] = df['original_duration'].gt(0).fillna(False)
df['valid_actual_duration'] = df['at_completion_duration'].ge(0).fillna(False)
df['has_actual_finish'] = df['actual_finish_parsed'].notna()
df['has_actual_start'] = df['actual_start_parsed'].notna()
df['marked_complete'] = df['physical_percent_complete'].eq(100).fillna(False)
df['valid_date_order'] = (
    df['has_actual_start'] & df['has_actual_finish'] &
    (df['actual_finish_parsed'] >= df['actual_start_parsed'])
)

df['numeric_candidate'] = df['valid_original'] & df['valid_actual_duration']
df['completed_candidate'] = (
    df['numeric_candidate'] & df['marked_complete'] &
    df['valid_date_order']
)

# Targets are defined only where the required duration values exist.
df['duration_target_days_unverified'] = df['at_completion_duration'].where(df['numeric_candidate'])
df['overrun_target_unverified'] = pd.Series(pd.NA, index=df.index, dtype='Int64')
mask = df['numeric_candidate']
df.loc[mask, 'overrun_target_unverified'] = (
    df.loc[mask, 'at_completion_duration'] > df.loc[mask, 'original_duration']
).astype('int64')

# Audit the supplied growth variable: likely a ratio for many records.
df['derived_growth_ratio'] = (
    (df['at_completion_duration'] - df['original_duration']) / df['original_duration']
).where(mask)
df['growth_difference_ratio'] = (
    df['reported_growth'] - df['derived_growth_ratio']
).abs()

print('Valid numeric duration candidates:', f"{df['numeric_candidate'].sum():,}")
print('Conservative completed candidates:', f"{df['completed_candidate'].sum():,}")
print('Zero at-completion duration among numeric candidates:',
      f"{(df['numeric_candidate'] & df['at_completion_duration'].eq(0)).sum():,}")
print('Growth ratio agrees to within 0.02:',
      f"{(mask & df['growth_difference_ratio'].le(0.02)).sum():,}")


### Validate numeric values, completion status, dates and target variable

I will check the numerical values, completion status, and dates to make sure the records are suitable for my machine learning model.

then i  will only create the target variable for records where the original estimated duration is greater than zero and the actual completion duration is zero or greater.

To create a more reliable dataset of completed activities, I will also check that Physical_Percent_Complete is equal to 100% and that an actual finish date has been recorded.

In [ ]:
num_cols = ['original_duration', 'at_completion_duration',
            'reported_growth', 'physical_percent_complete',
            'concurrent_activity_count']
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

for c in ['planned_start', 'planned_finish', 'actual_start', 'actual_finish', 'project_status_date']:
    df[c + '_parsed'] = pd.to_datetime(df[c], errors='coerce')

# Each flag is reported separately; records can fail multiple checks.
df['valid_original'] = df['original_duration'].gt(0).fillna(False)
df['valid_actual_duration'] = df['at_completion_duration'].ge(0).fillna(False)
df['has_actual_finish'] = df['actual_finish_parsed'].notna()
df['has_actual_start'] = df['actual_start_parsed'].notna()
df['marked_complete'] = df['physical_percent_complete'].eq(100).fillna(False)
df['valid_date_order'] = (
    df['has_actual_start'] & df['has_actual_finish'] &
    (df['actual_finish_parsed'] >= df['actual_start_parsed'])
)

df['numeric_candidate'] = df['valid_original'] & df['valid_actual_duration']
df['completed_candidate'] = (
    df['numeric_candidate'] & df['marked_complete'] &
    df['valid_date_order']
)

# Targets are defined only where the required duration values exist.
df['duration_target_days_unverified'] = df['at_completion_duration'].where(df['numeric_candidate'])
df['overrun_target_unverified'] = pd.Series(pd.NA, index=df.index, dtype='Int64')
mask = df['numeric_candidate']
df.loc[mask, 'overrun_target_unverified'] = (
    df.loc[mask, 'at_completion_duration'] > df.loc[mask, 'original_duration']
).astype('int64')

# Audit the supplied growth variable: likely a ratio for many records.
df['derived_growth_ratio'] = (
    (df['at_completion_duration'] - df['original_duration']) / df['original_duration']
).where(mask)
df['growth_difference_ratio'] = (
    df['reported_growth'] - df['derived_growth_ratio']
).abs()

print('Valid numeric duration candidates:', f"{df['numeric_candidate'].sum():,}")
print('Conservative completed candidates:', f"{df['completed_candidate'].sum():,}")
print('Zero at-completion duration among numeric candidates:',
      f"{(df['numeric_candidate'] & df['at_completion_duration'].eq(0)).sum():,}")
print('Growth ratio agrees to within 0.02:',
      f"{(mask & df['growth_difference_ratio'].le(0.02)).sum():,}")


## Data quality summary and exclusion reasons

In [ ]:
total = len(df)
checks = {
    'Missing or invalid original duration': int(df['original_duration'].isna().sum()),
    'Zero or negative original duration': int(df['original_duration'].le(0).sum()),
    'Missing or invalid at-completion duration': int(df['at_completion_duration'].isna().sum()),
    'Negative at-completion duration': int(df['at_completion_duration'].lt(0).sum()),
    'Not marked 100% complete': int((~df['marked_complete']).sum()),
    'Missing actual start date': int((~df['has_actual_start']).sum()),
    'Missing actual finish date': int((~df['has_actual_finish']).sum()),
    'Missing or reversed actual date range': int((~df['valid_date_order']).sum()),
    'Duplicate project/activity ID': int(df.duplicated(['project_id', 'activity_id']).sum()),
    'Numeric candidate': int(df['numeric_candidate'].sum()),
    'Completed candidate': int(df['completed_candidate'].sum())
}
quality = pd.DataFrame([{'check': k, 'count': v, 'percent_all_activities': round(v / total * 100, 2)}
                        for k, v in checks.items()])
display(quality)
quality.to_csv(OUTPUT_DIR / 'data_quality_summary.csv', index=False)

print('Missingness by extracted field (%)')
missing_pct = (df[RAW_COLUMNS].isna().mean() * 100).sort_values(ascending=False)
display(missing_pct.to_frame('missing_percent'))
missing_pct.rename('missing_percent').to_csv(OUTPUT_DIR / 'field_missingness.csv')


In [ ]:
# Create a provisional cleaned dataset
cleaned = df.loc[df['completed_candidate']].copy()

# Exclude duplicated IDs conservatively, retaining the first occurrence only.
duplicated = cleaned.duplicated(['project_id', 'activity_id'], keep='first')
print('Duplicate candidate rows removed:', int(duplicated.sum()))
cleaned = cleaned.loc[~duplicated].copy()

# Create user-friendly target names; their meaning is still provisional.
cleaned['actual_duration_target'] = cleaned['at_completion_duration']
cleaned['overrun_target'] = cleaned['overrun_target_unverified'].astype('int8')

# Preserve source fields needed for later review and auditing.
keep = [
    'project_id', 'activity_id', 'project_type', 'uk_region', 'gmm_topic',
    'task_type', 'original_duration', 'at_completion_duration',
    'physical_percent_complete', 'planned_start', 'planned_finish',
    'actual_start', 'actual_finish', 'concurrent_activity_count',
    'reported_growth', 'derived_growth_ratio',
    'actual_duration_target', 'overrun_target'
]
cleaned = cleaned[keep]
cleaned_file = OUTPUT_DIR / 'cleaned_completed_candidates.csv'
cleaned.to_csv(cleaned_file, index=False)
print('Saved:', cleaned_file)
print('Rows:', f'{len(cleaned):,}')
print('Project count:', cleaned['project_id'].nunique())
display(cleaned.head())

### Examine target distribution and project-level feasibility

In [ ]:
project_summary = cleaned.groupby('project_id', dropna=False).agg(
    candidate_activities=('activity_id', 'size'),
    overruns=('overrun_target', 'sum'),
    actual_duration_median=('actual_duration_target', 'median')
).reset_index()
project_summary['non_overruns'] = (
    project_summary['candidate_activities'] - project_summary['overruns']
)
project_summary.to_csv(OUTPUT_DIR / 'project_summary.csv', index=False)

print('Completed candidate projects:', len(project_summary))
print('Overrun distribution:')
print(cleaned['overrun_target'].value_counts(dropna=False).sort_index())
print('Target duration summary:')
display(cleaned['actual_duration_target'].describe(percentiles=[.5,.9,.95,.99]))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
cleaned['overrun_target'].value_counts().sort_index().plot.bar(ax=axes[0])
axes[0].set_title('Completed candidate class counts')
axes[0].set_xlabel('Overrun target (0 = no, 1 = yes)')
axes[0].set_ylabel('Activity records')
cleaned['actual_duration_target'].clip(upper=cleaned['actual_duration_target'].quantile(.99)).hist(bins=40, ax=axes[1])
axes[1].set_title('Actual duration (capped at 99th percentile for chart)')
axes[1].set_xlabel('Duration — units to verify')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'target_distribution.png', dpi=150)
plt.show()

### Feature leakage checklist

A field can be useful only if we can verify that its value existed at the prediction time


In [ ]:
feature_review = pd.DataFrame([
    ('original_duration', 'Potential predictor', 'Verify baseline was saved before activity start'),
    ('project_type', 'Potential predictor', 'Verify project classification known pre-start'),
    ('uk_region', 'Potential predictor', 'Verify known pre-start'),
    ('gmm_topic', 'Needs evidence', 'Could be a retrospectively generated classification'),
    ('task_type', 'Needs evidence', 'Check field definition; may contain task identifier'),
    ('concurrent_activity_count', 'Needs evidence', 'Could be calculated retrospectively'),
    ('planned_start', 'Needs evidence', 'Verify original not revised schedule date'),
    ('planned_finish', 'Needs evidence', 'Verify original not revised schedule date'),
    ('actual_start', 'Exclude as input', 'Usually not available before work begins'),
    ('actual_finish', 'Exclude as input', 'Outcome information'),
    ('physical_percent_complete', 'Exclude as input', 'Outcome/completion information'),
    ('reported_growth', 'Exclude as input', 'Derived from outcome'),
    ('derived_growth_ratio', 'Exclude as input', 'Derived from outcome'),
    ('at_completion_duration', 'Target only', 'Regression target source'),
    ('overrun_target', 'Target only', 'Classification target'),
], columns=['field', 'status', 'reason'])
display(feature_review)
feature_review.to_csv(OUTPUT_DIR / 'feature_review.csv', index=False)

In [ ]:
# Saving a reproducibility and limitations note

notes = {
    'dataset': 'JPF_Anonymised_Project_Data.json (Cambridge Apollo)',
    'source': 'https://doi.org/10.17863/CAM.53890',
    'input_zip_name': archive_path.name,
    'total_projects_extracted': int(stats['projects']),
    'total_activities_extracted': int(stats['activities']),
    'numeric_candidates': int(df['numeric_candidate'].sum()),
    'completed_candidates_after_duplicate_handling': int(len(cleaned)),
    'regression_target_source': 'At_Completion_Duration',
    'classification_definition': 'At_Completion_Duration > Original_Duration_Estimate(ODE)',
    'provisional_completed_definition': 'numeric candidate AND physical_percent_complete == 100 AND valid actual start/finish dates',
    'unresolved': [
        'Evidence original estimate was recorded before the activity began',
        'Whether At_Completion_Duration units and semantics are consistent',
        'Whether 100% completion and actual dates identify approved completed activities',
        'Whether candidate predictors were recorded before work began',
        'Difference between reported Duration_Growth_Percentage and recomputed ratio',
        'Potential extreme values/outliers and dataset-specific activity types'
    ],
    'no_train_test_split_yet': 'Project-level splitting will happen in notebook 03 after feature review',
    'no_model_fitted': True
}
with (OUTPUT_DIR / 'preprocessing_notes.json').open('w') as f:
    json.dump(notes, f, indent=2)
print(json.dumps(notes, indent=2))

{
  "dataset": "JPF_Anonymised_Project_Data.json (Cambridge Apollo)",
  "source": "https://doi.org/10.17863/CAM.53890",
  "input_zip_name": "JPF_Anonymised_Project_Data.7z",
  "total_projects_extracted": 299,
  "total_activities_extracted": 444156,
  "numeric_candidates": 293489,
  "completed_candidates_after_duplicate_handling": 160117,
  "regression_target_source": "At_Completion_Duration",
  "classification_definition": "At_Completion_Duration > Original_Duration_Estimate(ODE)",
  "provisional_completed_definition": "numeric candidate AND physical_percent_complete == 100 AND valid actual start/finish dates",
  "unresolved": [
    "Evidence original estimate was recorded before the activity began",
    "Whether At_Completion_Duration units and semantics are consistent",
    "Whether 100% completion and actual dates identify approved completed activities",
    "Whether candidate predictors were recorded before work began",
    "Difference between reported Duration_Growth_Percentage an

In [ ]:
from google.colab import files
import shutil

report_files = [
    'data_quality_summary.csv', 'field_missingness.csv', 'project_summary.csv',
    'feature_review.csv', 'preprocessing_notes.json', 'target_distribution.png'
]
report_dir = OUTPUT_DIR / 'shareable_reports'
report_dir.mkdir(exist_ok=True)
for name in report_files:
    shutil.copy2(OUTPUT_DIR / name, report_dir / name)
archive_path = shutil.make_archive('/content/construction_cleaning_reports', 'zip', report_dir)
print('Report ZIP:', archive_path)
print('Candidate dataset CSV:', cleaned_file)
files.download(archive_path)

Report ZIP: /content/construction_cleaning_reports.zip
Candidate dataset CSV: /content/construction_ml_preprocessing/cleaned_completed_candidates.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
import zipfile
from pathlib import Path

# Source zip file in your workspace
zip_source = Path('/content/JPF_Anonymised_Project_Data.json (1).zip')
extracted_dir = Path('/content/extracted')
extracted_dir.mkdir(parents=True, exist_ok=True)

if zip_source.exists():
    print(f"Extracting {zip_source}...")
    with zipfile.ZipFile(zip_source, 'r') as zip_ref:
        zip_ref.extractall(extracted_dir)
    print("Extraction complete.")

    # Check and locate the actual extracted JSON file
    extracted_files = list(extracted_dir.glob('**/*.json'))
    if extracted_files:
        target_path = Path('/content/extracted/JPF_Anonymised_Project_Data.json')
        if extracted_files[0] != target_path:
            extracted_files[0].rename(target_path)
        print(f"JSON prepared at: {target_path}")
    else:
        print("No JSON files found inside the ZIP.")
else:
    print(f"Zip file not found at {zip_source}. Please verify the filename.")

Extracting /content/JPF_Anonymised_Project_Data.json (1).zip...
Extraction complete.
JSON prepared at: /content/extracted/JPF_Anonymised_Project_Data.json


In [ ]:
import re
import csv
import math
import orjson
import pandas as pd
from collections import Counter

json_path = '/content/extracted/JPF_Anonymised_Project_Data.json'
OUTPUT_DIR = Path('/content/construction_ml_preprocessing')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_COLUMNS = [
    'project_id', 'activity_id', 'project_type', 'uk_region',
    'original_duration', 'at_completion_duration', 'reported_growth',
    'physical_percent_complete', 'planned_start', 'planned_finish',
    'actual_start', 'actual_finish', 'gmm_topic', 'task_type',
    'concurrent_activity_count', 'project_status_date'
]

PROJECT_OPEN = re.compile(rb'^ {12}"([^"]+)": \{\s*$')
PROJECT_CLOSE = re.compile(rb'^ {12}\},?\s*$')

def numeric(value):
    if value is None or isinstance(value, bool):
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

def extract_project(key, data, writer, stats):
    obj = orjson.loads(data)
    project_id = str(obj.get('Project_ID') or key)
    activities = obj.get('Activities') or {}
    if not isinstance(activities, dict):
        return
    stats['projects'] += 1
    for activity_key, a in activities.items():
        stats['activities'] += 1
        writer.writerow({
            'project_id': project_id,
            'activity_id': str(a.get('Task_ID') or activity_key),
            'project_type': obj.get('Project_Type'),
            'uk_region': obj.get('UK_Region'),
            'original_duration': numeric(a.get('Original_Duration_Estimate(ODE)')),
            'at_completion_duration': numeric(a.get('At_Completion_Duration')),
            'reported_growth': numeric(a.get('Duration_Growth_Percentage(Y_values)')),
            'physical_percent_complete': numeric(a.get('Physical_Percent_Complete')),
            'planned_start': a.get('Planned_Start_Date'),
            'planned_finish': a.get('Planned_Finish_Date'),
            'actual_start': a.get('Actual_Start_Date'),
            'actual_finish': a.get('Actual_Finish_Date'),
            'gmm_topic': a.get('GMM_Topic_Classification'),
            'task_type': a.get('Task_Type:'),
            'concurrent_activity_count': numeric(a.get('Concurrent_Activity_Count')),
            'project_status_date': obj.get('Status_Data_Date')
        })

raw_csv = OUTPUT_DIR / 'activity_audit.csv'
stats = Counter()

print("Processing JSON structure to raw CSV...")
with open(json_path, 'rb') as stream, \
     raw_csv.open('w', encoding='utf-8', newline='') as out:
    writer = csv.DictWriter(out, fieldnames=RAW_COLUMNS)
    writer.writeheader()
    current_key, parts = None, []
    in_projects = False
    for line in stream:
        if not in_projects:
            if line.strip() == b'"Projects": {':
                in_projects = True
            continue
        if current_key is None:
            match = PROJECT_OPEN.match(line)
            if match:
                current_key = match.group(1).decode('utf-8')
                parts = [b'{']
            elif line.startswith(b'      }') and line.strip() in (b'}', b'},'):
                break
        else:
            if PROJECT_CLOSE.match(line):
                parts.append(b'}')
                extract_project(current_key, b'\n'.join(parts), writer, stats)
                current_key, parts = None, []
            else:
                parts.append(line)

print('Projects parsed:', f"{stats['projects']:,}")
print('Activity records:', f"{stats['activities']:,}")

In [ ]:
print("Cleaning activities and building target variables...")
df = pd.read_csv(
    raw_csv,
    dtype={'project_id': 'string', 'activity_id': 'string',
           'project_type': 'string', 'uk_region': 'string',
           'gmm_topic': 'string', 'task_type': 'string'},
    low_memory=False
)

for c in ['original_duration', 'at_completion_duration', 'physical_percent_complete']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

for c in ['actual_start', 'actual_finish']:
    df[c + '_parsed'] = pd.to_datetime(df[c], errors='coerce')

df['valid_original'] = df['original_duration'].gt(0).fillna(False)
df['valid_actual_duration'] = df['at_completion_duration'].ge(0).fillna(False)
df['has_actual_finish'] = df['actual_finish_parsed'].notna()
df['has_actual_start'] = df['actual_start_parsed'].notna()
df['marked_complete'] = df['physical_percent_complete'].eq(100).fillna(False)
df['valid_date_order'] = (
    df['has_actual_start'] & df['has_actual_finish'] &
    (df['actual_finish_parsed'] >= df['actual_start_parsed'])
)

df['numeric_candidate'] = df['valid_original'] & df['valid_actual_duration']
df['completed_candidate'] = (
    df['numeric_candidate'] & df['marked_complete'] &
    df['valid_date_order']
)

df['overrun_target_unverified'] = pd.Series(pd.NA, index=df.index, dtype='Int64')
mask = df['numeric_candidate']
df.loc[mask, 'overrun_target_unverified'] = (
    df.loc[mask, 'at_completion_duration'] > df.loc[mask, 'original_duration']
).astype('int64')

cleaned = df.loc[df['completed_candidate']].copy()
cleaned = cleaned.drop_duplicates(['project_id', 'activity_id'], keep='first')

cleaned['actual_duration_target'] = cleaned['at_completion_duration']
cleaned['overrun_target'] = cleaned['overrun_target_unverified'].astype('int8')

keep = [
    'project_id', 'activity_id', 'project_type', 'uk_region', 'gmm_topic',
    'task_type', 'original_duration', 'at_completion_duration',
    'physical_percent_complete', 'planned_start', 'planned_finish',
    'actual_start', 'actual_finish', 'concurrent_activity_count',
    'reported_growth',
    'actual_duration_target', 'overrun_target'
]
cleaned = cleaned[keep]
cleaned_file = OUTPUT_DIR / 'cleaned_completed_candidates.csv'
cleaned.to_csv(cleaned_file, index=False)
print(f"Cleaned file generated with {len(cleaned):,} rows.")

In [ ]:
from google.colab import files
print("Downloading cleaned CSV file...")
files.download(str(cleaned_file))

In [13]:
# Fix escaping of quotes in regular expressions to ensure error-free script generation
import sys
import os
from pathlib import Path

OUTPUT_DIR = Path("/content/construction_ml_preprocessing")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Correctly escaped script generator for parsing
with open('parse.py', 'w') as f:
    f.write('''import re, csv, math, orjson, pandas as pd, os
from pathlib import Path
from collections import Counter
json_path = "/content/extracted/JPF_Anonymised_Project_Data.json"
OUTPUT_DIR = Path("/content/construction_ml_preprocessing")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_COLUMNS = ["project_id", "activity_id", "project_type", "uk_region", "original_duration", "at_completion_duration", "reported_growth", "physical_percent_complete", "planned_start", "planned_finish", "actual_start", "actual_finish", "gmm_topic", "task_type", "concurrent_activity_count", "project_status_date"]
PROJECT_OPEN = re.compile(rb'^ {12}"([^"]+)": \{\s*$')
PROJECT_CLOSE = re.compile(rb'^ {12}\},?\s*$')
def numeric(value):
    if value is None or isinstance(value, bool): return None
    try: return float(value)
    except: return None
def extract_project(key, data, writer, stats):
    obj = orjson.loads(data)
    project_id = str(obj.get("Project_ID") or key)
    activities = obj.get("Activities") or {}
    if not isinstance(activities, dict): return
    stats["projects"] += 1
    for activity_key, a in activities.items():
        stats["activities"] += 1
        writer.writerow({"project_id": project_id, "activity_id": str(a.get("Task_ID") or activity_key), "project_type": obj.get("Project_Type"), "uk_region": obj.get("UK_Region"), "original_duration": numeric(a.get("Original_Duration_Estimate(ODE)")), "at_completion_duration": numeric(a.get("At_Completion_Duration")), "reported_growth": numeric(a.get("Duration_Growth_Percentage(Y_values)")), "physical_percent_complete": numeric(a.get("Physical_Percent_Complete")), "planned_start": a.get("Planned_Start_Date"), "planned_finish": a.get("Planned_Finish_Date"), "actual_start": a.get("Actual_Start_Date"), "actual_finish": a.get("Actual_Finish_Date"), "gmm_topic": a.get("GMM_Topic_Classification"), "task_type": a.get("Task_Type:"), "concurrent_activity_count": numeric(a.get("Concurrent_Activity_Count")), "project_status_date": obj.get("Status_Data_Date")})
raw_csv = OUTPUT_DIR / "activity_audit.csv"
stats = Counter()
with open(json_path, "rb") as stream, raw_csv.open("w", encoding="utf-8", newline="") as out:
    writer = csv.DictWriter(out, fieldnames=RAW_COLUMNS)
    writer.writeheader()
    current_key, parts, in_projects = None, [], False
    for line in stream:
        if not in_projects:
            if line.strip() == b'"Projects": {': in_projects = True
            continue
        if current_key is None:
            match = PROJECT_OPEN.match(line)
            if match:
                current_key = match.group(1).decode("utf-8")
                parts = [b"{"]
            elif line.startswith(b"      }") and line.strip() in (b"}", b"},"):
                break
        else:
            if PROJECT_CLOSE.match(line):
                parts.append(b"}")
                extract_project(current_key, b"\\n".join(parts), writer, stats)
                current_key, parts = None, []
            else: parts.append(line)
print("Projects parsed:", stats["projects"])
print("Activity records:", stats["activities"])
''')

with open('clean.py', 'w') as f:
    f.write('''import pandas as pd
from pathlib import Path
OUTPUT_DIR = Path("/content/construction_ml_preprocessing")
raw_csv = OUTPUT_DIR / "activity_audit.csv"
df = pd.read_csv(raw_csv, dtype={"project_id": "string", "activity_id": "string", "project_type": "string", "uk_region": "string", "gmm_topic": "string", "task_type": "string"}, low_memory=False)
for c in ["original_duration", "at_completion_duration", "physical_percent_complete"]: df[c] = pd.to_numeric(df[c], errors="coerce")
for c in ["actual_start", "actual_finish"]: df[c + "_parsed"] = pd.to_datetime(df[c], errors="coerce")
df["valid_original"] = df["original_duration"].gt(0).fillna(False)
df["valid_actual_duration"] = df["at_completion_duration"].ge(0).fillna(False)
df["has_actual_finish"] = df["actual_finish_parsed"].notna()
df["has_actual_start"] = df["actual_start_parsed"].notna()
df["marked_complete"] = df["physical_percent_complete"].eq(100).fillna(False)
df["valid_date_order"] = (df["has_actual_start"] & df["has_actual_finish"] & (df["actual_finish_parsed"] >= df["actual_start_parsed"]))
df["numeric_candidate"] = df["valid_original"] & df["valid_actual_duration"]
df["completed_candidate"] = (df["numeric_candidate"] & df["marked_complete"] & df["valid_date_order"])
df["overrun_target_unverified"] = pd.Series(pd.NA, index=df.index, dtype="Int64")
mask = df["numeric_candidate"]
df.loc[mask, "overrun_target_unverified"] = (df.loc[mask, "at_completion_duration"] > df.loc[mask, "original_duration"]).astype("int64")
cleaned = df.loc[df["completed_candidate"]].copy()
cleaned = cleaned.drop_duplicates(["project_id", "activity_id"], keep="first")
cleaned["actual_duration_target"] = cleaned["at_completion_duration"]
cleaned["overrun_target"] = cleaned["overrun_target_unverified"].astype("int8")
keep = ["project_id", "activity_id", "project_type", "uk_region", "gmm_topic", "task_type", "original_duration", "at_completion_duration", "physical_percent_complete", "planned_start", "planned_finish", "actual_start", "actual_finish", "concurrent_activity_count", "reported_growth", "actual_duration_target", "overrun_target"]
cleaned = cleaned[keep]
cleaned_file = OUTPUT_DIR / "cleaned_completed_candidates.csv"
cleaned.to_csv(cleaned_file, index=False)
print("Cleaned file generated with", len(cleaned), "rows.")
''')

print("Executing Step 1...")
!python parse.py
print("Executing Step 2...")
!python clean.py
print("Executing Step 3 (Download)...")
from google.colab import files
files.download('/content/construction_ml_preprocessing/cleaned_completed_candidates.csv')

<>:18: SyntaxWarning: invalid escape sequence '\{'
<>:18: SyntaxWarning: invalid escape sequence '\{'
/tmp/ipykernel_839/2751437561.py:18: SyntaxWarning: invalid escape sequence '\{'
  PROJECT_OPEN = re.compile(rb'^ {12}"([^"]+)": \{\s*$')


Executing Step 1...
Projects parsed: 299
Activity records: 444156
Executing Step 2...
Cleaned file generated with 160117 rows.
Executing Step 3 (Download)...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>